
# AU non-AU filter with cuDF (GPU) / 使用 cuDF（GPU）的非澳大利亚数据过滤

This notebook uses **cuDF** to filter **only one file**:

```python
data/AUTokens50/part_0.parquet
```

目标：
- 使用 **GPU / cuDF** 加速
- 只处理 `data/AUTokens50/part_0.parquet`
- 使用**强特征规则**识别明显的非澳大利亚数据
- 输出：
  - removed rows database / 被删除数据数据库
  - kept rows / 保留数据
  - summary by rule / 按规则统计

**Important / 注意**  
This notebook assumes your environment already has a working RAPIDS + cuDF installation.  
本 notebook 假设你的环境已经正确安装了 RAPIDS + cuDF。


In [2]:

# ============================================================
# 0) Imports / 导入
# ============================================================
import os
import json
import re
from pathlib import Path

import pandas as pd

try:
    import cudf
except Exception as e:
    raise ImportError(
        "cuDF is not available in this environment. "
        "Please install RAPIDS/cuDF first, then rerun this notebook."
    ) from e

print("pandas version:", pd.__version__)
print("cuDF version:", cudf.__version__)


pandas version: 2.3.3
cuDF version: 26.04.000


In [3]:

# ============================================================
# 1) Config / 配置
# ============================================================
INPUT_FILE = "data/AUTokens50/part_0.parquet"

TEXT_COL = "text"
URL_COL = "url"
ID_COL = "id"

OUTPUT_ROOT = Path("./au_filter_outputs_cudf_part0")
REMOVED_DIR = OUTPUT_ROOT / "removed_rows"
KEPT_DIR = OUTPUT_ROOT / "kept_rows"
REPORT_DIR = OUTPUT_ROOT / "reports"

for d in [OUTPUT_ROOT, REMOVED_DIR, KEPT_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

WRITE_KEPT = True

print("INPUT_FILE exists:", Path(INPUT_FILE).exists())
print("OUTPUT_ROOT:", OUTPUT_ROOT.resolve())


INPUT_FILE exists: True
OUTPUT_ROOT: /home/jovyan/au_filter_outputs_cudf_part0


In [4]:

# ============================================================
# 2) Rules / 规则
# ============================================================
RULES = []

def add_rule(name, country, pattern, source="text", description=""):
    RULES.append({
        "name": name,
        "country": country,
        "pattern": pattern,
        "source": source,
        "description": description,
    })

# ---------- UK ----------
add_rule(
    "uk_domain_strong",
    "UK",
    r"(?:https?://)?(?:[\w-]+\.)+(?:gov\.uk|nhs\.uk|ac\.uk)\b",
    source="url_or_text",
    description="Strong UK domain"
)
add_rule(
    "uk_postcode_address",
    "UK",
    r"\b(?:address|postcode|post\s*code|contact|located at|office|campus)\b.{0,80}\b([A-Z]{1,2}\d[A-Z\d]?\s*\d[A-Z]{2})\b|\b([A-Z]{1,2}\d[A-Z\d]?\s*\d[A-Z]{2})\b.{0,80}\b(?:england|scotland|wales|united kingdom|uk)\b",
    source="text",
    description="UK postcode in address context"
)

# ---------- Canada ----------
add_rule(
    "canada_domain_strong",
    "Canada",
    r"(?:https?://)?(?:[\w-]+\.)+(?:gc\.ca)\b",
    source="url_or_text",
    description="Strong Canadian government domain"
)
add_rule(
    "canada_postal_code_address",
    "Canada",
    r"\b(?:address|postal code|postcode|contact|located at|office)\b.{0,80}\b([A-Z]\d[A-Z][ -]?\d[A-Z]\d)\b|\b([A-Z]\d[A-Z][ -]?\d[A-Z]\d)\b.{0,80}\b(?:canada|ontario|quebec|british columbia|alberta|manitoba|saskatchewan|nova scotia|new brunswick)\b",
    source="text",
    description="Canadian postal code in address context"
)

# ---------- New Zealand ----------
add_rule(
    "nz_domain_strong",
    "New Zealand",
    r"(?:https?://)?(?:[\w-]+\.)+(?:govt\.nz|ac\.nz|school\.nz)\b",
    source="url_or_text",
    description="Strong NZ domain"
)
add_rule(
    "nz_postcode_address",
    "New Zealand",
    r"\b(?:address|postcode|postal code|contact|located at|office)\b.{0,80}\b(\d{4})\b.{0,40}\b(?:new zealand|nz|auckland|wellington|christchurch|hamilton|dunedin)\b|\b(?:new zealand|nz|auckland|wellington|christchurch|hamilton|dunedin)\b.{0,80}\b(\d{4})\b",
    source="text",
    description="NZ postcode in address context"
)

# ---------- United States ----------
add_rule(
    "us_zip_with_state",
    "United States",
    r"\b\d{5}(?:-\d{4})?\b.{0,40}\b(?:AL|AK|AZ|AR|CA|CO|CT|DE|FL|GA|HI|IA|ID|IL|IN|KS|KY|LA|MA|MD|ME|MI|MN|MO|MS|MT|NC|ND|NE|NH|NJ|NM|NV|NY|OH|OK|OR|PA|RI|SC|SD|TN|TX|UT|VA|VT|WA|WI|WV|DC)\b|\b(?:AL|AK|AZ|AR|CA|CO|CT|DE|FL|GA|HI|IA|ID|IL|IN|KS|KY|LA|MA|MD|ME|MI|MN|MO|MS|MT|NC|ND|NE|NH|NJ|NM|NV|NY|OH|OK|OR|PA|RI|SC|SD|TN|TX|UT|VA|VT|WA|WI|WV|DC)\b.{0,40}\b\d{5}(?:-\d{4})?\b",
    source="text",
    description="US ZIP near state abbreviation"
)
add_rule(
    "us_zip_with_address_terms",
    "United States",
    r"\b(?:zip|zipcode|zip code|address|suite|ste\.?|apt\.?|po box)\b.{0,60}\b\d{5}(?:-\d{4})?\b|\b\d{5}(?:-\d{4})?\b.{0,60}\b(?:usa|united states)\b",
    source="text",
    description="US ZIP in address context"
)
add_rule(
    "us_domain_strong_review",
    "United States",
    r"(?:https?://)?(?:[\w-]+\.)+(?:mil)\b",
    source="url_or_text",
    description="Very strong US military domain"
)

# ---------- Phone + country context ----------
add_rule(
    "uk_phone_plus_country",
    "UK",
    r"\+44[\d\s\-\(\)]{6,}\b.{0,60}\b(?:united kingdom|uk|england|scotland|wales)\b|\b(?:united kingdom|uk|england|scotland|wales)\b.{0,60}\+44[\d\s\-\(\)]{6,}",
    source="text",
    description="UK phone with UK context"
)
add_rule(
    "ca_phone_plus_country",
    "Canada",
    r"\+1[\d\s\-\(\)]{6,}\b.{0,60}\b(?:canada|ontario|quebec|british columbia|alberta)\b|\b(?:canada|ontario|quebec|british columbia|alberta)\b.{0,60}\+1[\d\s\-\(\)]{6,}",
    source="text",
    description="Canada phone with Canada context"
)
add_rule(
    "nz_phone_plus_country",
    "New Zealand",
    r"\+64[\d\s\-\(\)]{5,}\b.{0,60}\b(?:new zealand|nz)\b|\b(?:new zealand|nz)\b.{0,60}\+64[\d\s\-\(\)]{5,}",
    source="text",
    description="NZ phone with NZ context"
)

print("Number of rules / 规则数量:", len(RULES))
pd.DataFrame(RULES)


Number of rules / 规则数量: 12


,name,country,pattern,source,description
0,uk_domain_strong,UK,(?:https?://)?(?:[\w-]+\.)+(?:gov\.uk|nhs\.uk|...,url_or_text,Strong UK domain
1,uk_postcode_address,UK,\b(?:address|postcode|post\s*code|contact|loca...,text,UK postcode in address context
2,canada_domain_strong,Canada,(?:https?://)?(?:[\w-]+\.)+(?:gc\.ca)\b,url_or_text,Strong Canadian government domain
3,canada_postal_code_address,Canada,\b(?:address|postal code|postcode|contact|loca...,text,Canadian postal code in address context
4,nz_domain_strong,New Zealand,(?:https?://)?(?:[\w-]+\.)+(?:govt\.nz|ac\.nz|...,url_or_text,Strong NZ domain
5,nz_postcode_address,New Zealand,\b(?:address|postcode|postal code|contact|loca...,text,NZ postcode in address context
6,us_zip_with_state,United States,"\b\d{5}(?:-\d{4})?\b.{0,40}\b(?:AL|AK|AZ|AR|CA...",text,US ZIP near state abbreviation
7,us_zip_with_address_terms,United States,\b(?:zip|zipcode|zip code|address|suite|ste\.?...,text,US ZIP in address context
8,us_domain_strong_review,United States,(?:https?://)?(?:[\w-]+\.)+(?:mil)\b,url_or_text,Very strong US military domain
9,uk_phone_plus_country,UK,"\+44[\d\s\-\(\)]{6,}\b.{0,60}\b(?:united kingd...",text,UK phone with UK context


In [5]:

# ============================================================
# 3) Load parquet with cuDF / 使用 cuDF 读取 parquet
# ============================================================
gdf = cudf.read_parquet(INPUT_FILE)
print("Loaded rows, cols:", gdf.shape)
print("Columns:", list(gdf.columns))

assert TEXT_COL in gdf.columns, f"Missing text column: {TEXT_COL}"
assert URL_COL in gdf.columns, f"Missing url column: {URL_COL}"
assert ID_COL in gdf.columns, f"Missing id column: {ID_COL}"

gdf[[TEXT_COL, URL_COL, ID_COL]].head()


Loaded rows, cols: (278106, 9)
Columns: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count']


,text,url,id
664,"Justine Davies –, Monday, January, 31, 2011, (...",http://blogs.news.com.au/moneystuff/index.php/...,<urn:uuid:0f3c1f06-4af1-4da0-96f9-91a7473f205f>
5457,Interview: Jenny Macpherson\n- by Rowena Scott...,http://www.bicycles.net.au/2010/10/interview-j...,<urn:uuid:12bb0bca-ddf8-4522-97ee-5583b8c93b93>
5459,The foundations for successful riding\n19 post...,http://www.bicycles.net.au/forums/viewtopic.ph...,<urn:uuid:74766fcf-d5eb-44fd-b925-01bd98098d70>
8507,photos by Carlo Ledesma\nWhen God was creating...,http://www.kluster.com.au/issuefive/bees/,<urn:uuid:161ec30a-6da3-4d83-be97-32d4d9684b0c>
9656,Joanne Harris is apparently as formidable a Yo...,http://www.northerndailyleader.com.au/story/97...,<urn:uuid:2058b3ba-a9ad-4d81-80ab-4ce6b79c5d9e>


In [6]:

# ============================================================
# 4) GPU filtering / GPU 过滤
# ============================================================
gdf["_drop_non_au"] = False
gdf["_drop_rule"] = None
gdf["_drop_country"] = None

text_s = gdf[TEXT_COL].fillna("").astype("str")
url_s = gdf[URL_COL].fillna("").astype("str")
combined_s = text_s + "\n" + url_s

for i, rule in enumerate(RULES, start=1):
    if rule["source"] == "text":
        target = text_s
    elif rule["source"] == "url":
        target = url_s
    else:
        target = combined_s

    match_mask = target.str.contains(rule["pattern"], regex=True)
    apply_mask = (~gdf["_drop_non_au"]) & match_mask

    n_match = int(apply_mask.sum())
    print(f"[{i}/{len(RULES)}] {rule['name']} -> newly matched rows: {n_match}")

    if n_match > 0:
        gdf.loc[apply_mask, "_drop_non_au"] = True
        gdf.loc[apply_mask, "_drop_rule"] = rule["name"]
        gdf.loc[apply_mask, "_drop_country"] = rule["country"]

removed_gdf = gdf[gdf["_drop_non_au"]]
kept_gdf = gdf[~gdf["_drop_non_au"]]

print("Total rows:", len(gdf))
print("Removed rows:", len(removed_gdf))
print("Kept rows:", len(kept_gdf))
print("Removal rate:", len(removed_gdf) / max(len(gdf), 1))


[1/12] uk_domain_strong -> newly matched rows: 127
[2/12] uk_postcode_address -> newly matched rows: 9
[3/12] canada_domain_strong -> newly matched rows: 8
[4/12] canada_postal_code_address -> newly matched rows: 0
[5/12] nz_domain_strong -> newly matched rows: 22
[6/12] nz_postcode_address -> newly matched rows: 42
[7/12] us_zip_with_state -> newly matched rows: 268
[8/12] us_zip_with_address_terms -> newly matched rows: 6
[9/12] us_domain_strong_review -> newly matched rows: 3
[10/12] uk_phone_plus_country -> newly matched rows: 6
[11/12] ca_phone_plus_country -> newly matched rows: 0
[12/12] nz_phone_plus_country -> newly matched rows: 0
Total rows: 278106
Removed rows: 491
Kept rows: 277615
Removal rate: 0.001765513868812611


In [7]:

# ============================================================
# 5) Save outputs / 保存输出
# ============================================================
stem = Path(INPUT_FILE).stem

removed_path = REMOVED_DIR / f"{stem}__removed.parquet"
kept_path = KEPT_DIR / f"{stem}__kept.parquet"

removed_gdf.to_parquet(removed_path, index=False)
if WRITE_KEPT:
    kept_gdf.to_parquet(kept_path, index=False)

print("Removed parquet:", removed_path.resolve())
if WRITE_KEPT:
    print("Kept parquet:", kept_path.resolve())


Removed parquet: /home/jovyan/au_filter_outputs_cudf_part0/removed_rows/part_0__removed.parquet
Kept parquet: /home/jovyan/au_filter_outputs_cudf_part0/kept_rows/part_0__kept.parquet


In [8]:
# ============================================================
# 5b) Convert removed parquet to CSV / 将 removed parquet 转成 CSV
# ============================================================
import pandas as pd
from pathlib import Path

# 这里直接复用你上面已经生成的 removed_path
removed_csv_path = removed_path.with_suffix(".csv")

# 读取 parquet
removed_df = pd.read_parquet(removed_path)

# 可选：把空值填成空字符串，导出 CSV 更直观
removed_df = removed_df.fillna("")

# 写出 CSV
removed_df.to_csv(removed_csv_path, index=False, encoding="utf-8-sig")

print("Removed CSV:", removed_csv_path.resolve())
print("Rows:", len(removed_df))
print("Columns:", list(removed_df.columns))

Removed CSV: /home/jovyan/au_filter_outputs_cudf_part0/removed_rows/part_0__removed.csv
Rows: 491
Columns: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count', '_drop_non_au', '_drop_rule', '_drop_country']


In [9]:

# ============================================================
# 6) Reports / 统计报告
# ============================================================
summary_df = pd.DataFrame([{
    "source_file": INPUT_FILE,
    "n_total": int(len(gdf)),
    "n_removed": int(len(removed_gdf)),
    "n_kept": int(len(kept_gdf)),
    "removal_rate": float(len(removed_gdf) / max(len(gdf), 1)),
}])
summary_path = REPORT_DIR / "filter_summary_by_file.csv"
summary_df.to_csv(summary_path, index=False)

rule_summary_pdf = (
    removed_gdf["_drop_rule"]
    .value_counts(dropna=False)
    .to_pandas()
    .reset_index()
)
rule_summary_pdf.columns = ["rule", "count"]
rule_summary_pdf = rule_summary_pdf.sort_values("count", ascending=False)
rule_summary_path = REPORT_DIR / "filter_summary_by_rule.csv"
rule_summary_pdf.to_csv(rule_summary_path, index=False)

print(summary_df)
print(rule_summary_pdf.head(20))
print("Summary by file:", summary_path.resolve())
print("Summary by rule:", rule_summary_path.resolve())


                      source_file  n_total  n_removed  n_kept  removal_rate
0  data/AUTokens50/part_0.parquet   278106        491  277615      0.001766
                        rule  count
0          us_zip_with_state    268
1           uk_domain_strong    127
2        nz_postcode_address     42
3           nz_domain_strong     22
4        uk_postcode_address      9
5       canada_domain_strong      8
6  us_zip_with_address_terms      6
7      uk_phone_plus_country      6
8    us_domain_strong_review      3
Summary by file: /home/jovyan/au_filter_outputs_cudf_part0/reports/filter_summary_by_file.csv
Summary by rule: /home/jovyan/au_filter_outputs_cudf_part0/reports/filter_summary_by_rule.csv


In [ ]:

# ============================================================
# 7) Removed rows database / 被删除数据数据库
# ============================================================
removed_db_parquet = REPORT_DIR / "removed_rows_database.parquet"
removed_gdf.to_parquet(removed_db_parquet, index=False)

removed_db_csv = REPORT_DIR / "removed_rows_database.csv"
removed_gdf.to_pandas().to_csv(removed_db_csv, index=False)

print("Removed rows database parquet:", removed_db_parquet.resolve())
print("Removed rows database csv:", removed_db_csv.resolve())
print("Done.")



## Notes / 说明

This cuDF version is much faster than the earlier row-by-row Python loop because it uses vectorized GPU string matching.  
这一版比之前逐行 Python 循环快很多，因为它使用了 GPU 上的向量化字符串匹配。

### Output files / 输出文件
- `au_filter_outputs_cudf_part0/removed_rows/part_0__removed.parquet`
- `au_filter_outputs_cudf_part0/kept_rows/part_0__kept.parquet`
- `au_filter_outputs_cudf_part0/reports/filter_summary_by_file.csv`
- `au_filter_outputs_cudf_part0/reports/filter_summary_by_rule.csv`
- `au_filter_outputs_cudf_part0/reports/removed_rows_database.parquet`
- `au_filter_outputs_cudf_part0/reports/removed_rows_database.csv`

### Important limitation / 重要限制
This notebook intentionally uses **strong rules only**.  
It does **not** try to remove every foreign topic mention.  
这个 notebook 只用**强规则**，不会因为文本提到国外新闻就误删。


In [3]:
!nvidia-smi

Tue Apr 21 05:28:22 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.288.01             Driver Version: 535.288.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA L4                      Off | 00000000:25:00.0 Off |                    0 |
| N/A   46C    P8              17W /  72W |      0MiB / 23034MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [4]:
!pip install cudf-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 45.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 35.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 MB 106.5 MB/s  0:00:060:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 117.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 115.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 124.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 104.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 24.1 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 92.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.9/30.9 MB 105.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 104.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 109.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━

In [1]:
import cudf
print(cudf.__version__)

26.04.000


In [ ]:
## Alternative Method: Quarantine-Based Filtering with Australian Signal Recovery

### 1. Overview

This is an alternative method I considered, but I have not implemented it in code yet.

We adopt a conservative two-stage filtering strategy to remove non-Australian data while minimizing false positives.

Instead of directly deleting suspected samples, we:

1. Flag high-confidence non-Australian data.
2. Move them into a quarantine dataset.
3. Search for Australian signals within the quarantine set.
4. Perform manual review for ambiguous cases.

This design helps prevent the accidental removal of valid Australian-related content.

---

### 2. Stage 1: Foreign Strong-Signal Detection

Likely non-Australian samples can be identified using high-precision structural rules, including:

- **Domains:** `.gov.uk`, `.gc.ca`, `.govt.nz`, `.mil`
- **Postcodes:** UK, Canada, and New Zealand postcode formats with context
- **Address patterns:** for example, US ZIP code + state abbreviation
- **Phone numbers:** `+44`, `+1`, `+64`
- **Public service terms:** IRS, HMRC, CRA, etc.
- **Education systems:** GCSE, SAT, NCEA

All matched samples would be moved into a quarantine dataset, rather than being deleted immediately.

---

### 3. Stage 2: Australian Signal Recovery

The quarantine dataset would then be re-evaluated using Australian-specific signals, such as:

- `.gov.au`, `.edu.au`
- State + postcode patterns, for example `NSW 2000`
- `+61` phone numbers
- Terms such as Medicare, Centrelink, and ATO
- Australian cities, universities, and media sources

Each sample would be assigned an Australian signal score.

---

### 4. Recovery and Review

Samples with strong Australian signals would be treated as **recovery candidates**.

Samples without Australian recovery signals would remain as **final removal candidates**.

Samples containing both foreign and Australian signals would be flagged for **manual review**, ensuring that mixed-context data is handled carefully.

---

### 5. Final Dataset

The final dataset would consist of:

- Clean data that was never flagged
- Optionally recovered samples after manual review

---

### 6. Key Advantages

This method has several advantages:

- High-precision filtering using structural rules
- Reduced false positives through the recovery stage
- Interpretable and rule-based decision process

---

### 7. Summary

This pipeline introduces a quarantine-and-recovery framework that balances data cleaning with data preservation. It is suitable for building high-quality Australian-focused training datasets while reducing the risk of accidentally removing useful Australian-related content.